In [4]:
from google.colab import drive
drive.mount('/content/drive')

!pip install mlflow dagshub xgboost imbalanced-learn scikit-learn -q

!git clone https://github.com/Dimuthu21/fraud-risk-platform.git
import sys
sys.path.append('/content/fraud-risk-platform')

from src.features import save_train_columns

Mounted at /content/drive
fatal: destination path 'fraud-risk-platform' already exists and is not an empty directory.


Connect MLflow to DagsHub

In [2]:
import dagshub
dagshub.init(repo_owner='Dimuthu21', repo_name='fraud-risk-platform', mlflow=True)
import mlflow

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0a5d9daf-6faf-4a1b-86e8-5b558bdf4cb8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f11267ab350dfadc14ed140dd9b41651966429eb99b78837e8dd822878e02832




Accessing as Dimuthu21

Initialized MLflow to track repo "Dimuthu21/fraud-risk-platform"

Repository Dimuthu21/fraud-risk-platform initialized!

Load processed splits + save the training column schema

In [5]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/fraud-risk-platform/processed/train.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/fraud-risk-platform/processed/val.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/fraud-risk-platform/processed/test.csv')

save_train_columns(train_df)  # writes models/train_columns.json — locked in now for the API later

Split features/target

In [6]:
target_col = 'fraud_bool'
feature_cols = [c for c in train_df.columns if c != target_col]

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]

Evaluation helper (PR-AUC, not accuracy, is your headline metric)

In [7]:
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score, recall_score, precision_score

def evaluate(model, X, y, threshold=0.5):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        "pr_auc": average_precision_score(y, probs),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
    }

Baseline: Logistic Regression, no imbalance handling

In [8]:
from sklearn.linear_model import LogisticRegression

# quick one-off imputation of -1 sentinels, just for this baseline (see reasoning above)
X_train_lr = X_train.copy()
X_val_lr = X_val.copy()
sentinel_cols = ['prev_address_months_count', 'bank_months_count', 'current_address_months_count',
                  'session_length_in_minutes', 'device_distinct_emails_8w']
for col in sentinel_cols:
    median_val = X_train_lr.loc[X_train_lr[col] != -1, col].median()
    X_train_lr[col] = X_train_lr[col].replace(-1, median_val)
    X_val_lr[col] = X_val_lr[col].replace(-1, median_val)

with mlflow.start_run(run_name="logreg_baseline_no_weighting"):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_lr, y_train)
    metrics = evaluate(model, X_val_lr, y_val)
    mlflow.log_params({"model": "LogisticRegression", "class_weight": "none"})
    mlflow.log_metrics(metrics)
    print(metrics)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'pr_auc': np.float64(0.05665587711423252), 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
🏃 View run logreg_baseline_no_weighting at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0/runs/b0c0760752334cfd9c5e42248af9f435
🧪 View experiment at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0


Logistic Regression with class weighting

In [9]:
with mlflow.start_run(run_name="logreg_balanced"):
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_lr, y_train)
    metrics = evaluate(model, X_val_lr, y_val)
    mlflow.log_params({"model": "LogisticRegression", "class_weight": "balanced"})
    mlflow.log_metrics(metrics)
    print(metrics)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'pr_auc': np.float64(0.06078348026083579), 'precision': 0.02742390619435084, 'recall': 0.6903448275862069, 'f1': 0.052752233142736686}
🏃 View run logreg_balanced at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0/runs/fb88581cf11b4edabc79d3c950dce17d
🧪 View experiment at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0


Random Forest, class-weighted

In [10]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="random_forest_balanced"):
    model = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    metrics = evaluate(model, X_val, y_val)
    mlflow.log_params({"model": "RandomForest", "n_estimators": 200, "class_weight": "balanced"})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, "model")
    print(metrics)

2026/08/13 10:16:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


{'pr_auc': np.float64(0.11635125736830407), 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
🏃 View run random_forest_balanced at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0/runs/5e6576016b414a6085f03a24beba6a0c
🧪 View experiment at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0


XGBoost

In [11]:
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

with mlflow.start_run(run_name="xgboost_scaled"):
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, eval_metric='aucpr', n_jobs=-1, random_state=42
    )
    model.fit(X_train, y_train)
    metrics = evaluate(model, X_val, y_val)
    mlflow.log_params({"model": "XGBoost", "n_estimators": 300, "max_depth": 6, "scale_pos_weight": round(scale_pos_weight,2)})
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(model, "model")
    print(metrics)

scale_pos_weight: 96.53269537480064


2026/08/13 10:19:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


{'pr_auc': np.float64(0.15886406816713305), 'precision': 0.06685166498486378, 'recall': 0.7310344827586207, 'f1': 0.1225008667514157}
🏃 View run xgboost_scaled at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0/runs/71ef732d9a3b404588e1fbef57bd6600
🧪 View experiment at: https://dagshub.com/Dimuthu21/fraud-risk-platform.mlflow/#/experiments/0


In [12]:
import os
os.makedirs('/content/fraud-risk-platform/models', exist_ok=True)